# TA-004D — Grad-CAM: Visualización de Activaciones

**Objetivo:** Verificar que los 3 modelos CNN augmentados identifican las zonas anatómicas correctas en las radiografías.

**Setup:** 3 modelos × 5 clases × 5 muestras aleatorias = 75 visualizaciones.

In [1]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
import wandb

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

BASE        = Path(r'E:\Taller Integrador I\ModelosIA\Radiografias')
CHECKPOINTS = Path(r'E:\Taller Integrador I\ModelosIA\modelos\checkpoints')
OUTPUT_DIR  = Path(r'E:\Taller Integrador I\ModelosIA\modelos\Notebooks\gradcam_outputs')
TEST_DIR    = BASE / 'dataset_split' / 'test'
MANIFESTS   = BASE / 'dataset_split' / 'manifests'
OUTPUT_DIR.mkdir(exist_ok=True)

CLASSES     = ['alveolar_pattern', 'bronchial_pattern', 'pleural_effusion', 'cardiomegaly', 'no_finding']
NUM_CLASSES = len(CLASSES)
N_SAMPLES   = 5
DROPOUT     = 0.4
SEED        = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Config OK')

Device: cuda
Config OK


In [3]:
wandb.init(
    project='vetxray-cnn',
    entity='dbaylont1-antenor-orrego-private-university',
    name='TA-004D-GradCAM',
    config={
        'task': 'TA-004D',
        'modelos': ['ResNet-50', 'EfficientNet-B0', 'DenseNet-121'],
        'clases': 5,
        'muestras_por_clase': 5,
        'total_visualizaciones': 75
    },
    tags=['GradCAM', 'TA-004D', 'VetXRay', 'augmented']
)
print('W&B run:', wandb.run.url)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Pcuser\_netrc.
wandb: Currently logged in as: dbaylont1 (dbaylont1-antenor-orrego-private-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/dbaylont1-antenor-orrego-private-university/vetxray-cnn/runs/4eri75nr


In [4]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        self._fh = target_layer.register_forward_hook(self._fwd_hook)
        self._bh = target_layer.register_full_backward_hook(self._bwd_hook)

    def _fwd_hook(self, module, inp, out):
        self.activations = out.detach()

    def _bwd_hook(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx):
        self.model.eval()
        out = self.model(input_tensor)
        self.model.zero_grad()
        out[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = torch.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

    def remove(self):
        self._fh.remove()
        self._bh.remove()

print('GradCAM OK')

GradCAM OK


In [5]:
def load_resnet50():
    m = models.resnet50(weights=None)
    m.fc = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.fc.in_features, NUM_CLASSES))
    f = sorted(CHECKPOINTS.glob('resnet50_aug_best*.pth'),
               key=lambda p: int(p.stem.split('ep')[-1]))
    m.load_state_dict(torch.load(f[-1], map_location=DEVICE)['model_state_dict'])
    print(f'ResNet-50 cargado: {f[-1].name}')
    return m.to(DEVICE).eval(), m.layer4

def load_efficientnet():
    m = models.efficientnet_b0(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.classifier[1].in_features, NUM_CLASSES))
    f = sorted(CHECKPOINTS.glob('efficientnet_aug_best*.pth'),
               key=lambda p: int(p.stem.split('ep')[-1]))
    m.load_state_dict(torch.load(f[-1], map_location=DEVICE)['model_state_dict'])
    print(f'EfficientNet-B0 cargado: {f[-1].name}')
    return m.to(DEVICE).eval(), m.features[-1]

def load_densenet():
    m = models.densenet121(weights=None)
    m.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(m.classifier.in_features, NUM_CLASSES))
    f = sorted(CHECKPOINTS.glob('densenet_aug_best*.pth'),
               key=lambda p: int(p.stem.split('ep')[-1]))
    m.load_state_dict(torch.load(f[-1], map_location=DEVICE)['model_state_dict'])
    print(f'DenseNet-121 cargado: {f[-1].name}')
    return m.to(DEVICE).eval(), m.features.denseblock4

resnet,   resnet_layer = load_resnet50()
effnet,   effnet_layer = load_efficientnet()
densenet, dense_layer  = load_densenet()

MODELS = [
    ('ResNet-50',       resnet,   resnet_layer),
    ('EfficientNet-B0', effnet,   effnet_layer),
    ('DenseNet-121',    densenet, dense_layer),
]
print('Modelos cargados OK')

ResNet-50 cargado: resnet50_aug_best_ep11.pth
EfficientNet-B0 cargado: efficientnet_aug_best_ep25.pth
DenseNet-121 cargado: densenet_aug_best_ep15.pth
Modelos cargados OK


In [6]:
TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

def load_dicom_raw(path):
    dcm = pydicom.dcmread(str(path))
    arr = dcm.pixel_array.astype(np.float32)
    if hasattr(dcm, 'PhotometricInterpretation') and dcm.PhotometricInterpretation == 'MONOCHROME1':
        arr = arr.max() - arr
    p2, p98 = np.percentile(arr, 2), np.percentile(arr, 98)
    arr = np.clip(arr, p2, p98)
    arr = (arr - p2) / (p98 - p2 + 1e-8)
    img = Image.fromarray((arr * 255).astype(np.uint8))
    return img.resize((224, 224), Image.LANCZOS)

def img_to_tensor(pil_img):
    rgb = Image.merge('RGB', [pil_img, pil_img, pil_img])
    return TRANSFORM(rgb).unsqueeze(0).to(DEVICE)

def overlay_cam(pil_img, cam):
    img_np  = np.array(pil_img)
    cam_res = cv2.resize(cam, (224, 224))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_res), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    img_rgb = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
    return img_np, (0.5 * img_rgb + 0.5 * heatmap).astype(np.uint8)

df_test = pd.read_csv(MANIFESTS / 'test.csv')
print(f'Test set: {len(df_test)} imagenes')

def sample_class(cls, n=N_SAMPLES):
    sub = df_test[df_test['TAG'].str.contains(cls, na=False)]
    return sub.sample(min(n, len(sub)), random_state=SEED)

print('Loaders OK')

Test set: 940 imagenes
Loaders OK


In [7]:
for model_name, model, target_layer in MODELS:
    print(f'\n=== {model_name} ===')
    gcam = GradCAM(model, target_layer)

    for cls_idx, cls_name in enumerate(CLASSES):
        samples = sample_class(cls_name)
        n       = len(samples)
        fig, axes = plt.subplots(n, 2, figsize=(6, 3 * n))
        if n == 1:
            axes = [axes]
        fig.suptitle(f'{model_name} | {cls_name}', fontsize=12, fontweight='bold')

        for row, (_, sample) in enumerate(samples.iterrows()):
            pil_img      = load_dicom_raw(TEST_DIR / sample['FileName'])
            tensor       = img_to_tensor(pil_img)
            cam          = gcam.generate(tensor, cls_idx)
            raw, overlay = overlay_cam(pil_img, cam)

            axes[row][0].imshow(raw, cmap='gray')
            axes[row][0].set_title('Original', fontsize=9)
            axes[row][0].axis('off')

            axes[row][1].imshow(overlay)
            axes[row][1].set_title('Grad-CAM', fontsize=9)
            axes[row][1].axis('off')

        plt.tight_layout()
        safe_name = model_name.replace('-', '_').replace(' ', '_')
        fname     = OUTPUT_DIR / f'{safe_name}_{cls_name}.png'
        plt.savefig(fname, dpi=120, bbox_inches='tight')
        plt.close()
        wandb.log({f'{model_name}/{cls_name}': wandb.Image(str(fname), caption=f'{model_name} | {cls_name}')})
        print(f'  [{cls_name}] {n} muestras guardadas')

    gcam.remove()

print('\nTodas las visualizaciones generadas.')


=== ResNet-50 ===
  [alveolar_pattern] 5 muestras guardadas
  [bronchial_pattern] 5 muestras guardadas
  [pleural_effusion] 5 muestras guardadas
  [cardiomegaly] 5 muestras guardadas
  [no_finding] 5 muestras guardadas

=== EfficientNet-B0 ===
  [alveolar_pattern] 5 muestras guardadas
  [bronchial_pattern] 5 muestras guardadas
  [pleural_effusion] 5 muestras guardadas
  [cardiomegaly] 5 muestras guardadas
  [no_finding] 5 muestras guardadas

=== DenseNet-121 ===
  [alveolar_pattern] 5 muestras guardadas
  [bronchial_pattern] 5 muestras guardadas
  [pleural_effusion] 5 muestras guardadas
  [cardiomegaly] 5 muestras guardadas
  [no_finding] 5 muestras guardadas

Todas las visualizaciones generadas.


In [8]:
files = sorted(OUTPUT_DIR.glob('*.png'))
print(f'Total archivos generados: {len(files)}')
print(f'Carpeta: {OUTPUT_DIR}')
wandb.finish()
for f in files:
    print(f'  {f.name}')

Total archivos generados: 15
Carpeta: E:\Taller Integrador I\ModelosIA\modelos\Notebooks\gradcam_outputs


  DenseNet_121_alveolar_pattern.png
  DenseNet_121_bronchial_pattern.png
  DenseNet_121_cardiomegaly.png
  DenseNet_121_no_finding.png
  DenseNet_121_pleural_effusion.png
  EfficientNet_B0_alveolar_pattern.png
  EfficientNet_B0_bronchial_pattern.png
  EfficientNet_B0_cardiomegaly.png
  EfficientNet_B0_no_finding.png
  EfficientNet_B0_pleural_effusion.png
  ResNet_50_alveolar_pattern.png
  ResNet_50_bronchial_pattern.png
  ResNet_50_cardiomegaly.png
  ResNet_50_no_finding.png
  ResNet_50_pleural_effusion.png
